# Import Required Libraries and Create Clean Datasets

## Import Required Libraries

In [40]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib
import urllib.request
from bs4 import BeautifulSoup
import sqlite3
import sys
!{sys.executable} -m pip install "kagglehub[pandas-datasets]"
import kagglehub
# print("kagglehub works!")
from kagglehub import KaggleDatasetAdapter

## Webscrape the Hosts for Each Year

### Fetch and Locate Hosts Data

In [41]:
# Fetch HTML from Wikipedia page
hosts_url = "https://en.wikipedia.org/wiki/FIFA_World_Cup"

# Create request with custom User Agent to avoid having the request blocked
req = urllib.request.Request(hosts_url, headers={"User-Agent" : "Magic Browser"})

# Retrieve page content
con = urllib.request.urlopen(req)

# Parse the HTML using BeautifulSoup
soup = BeautifulSoup(con.read(), "html.parser")

# Find all tables with the class "wikitable"
wiki_tables = soup.find_all("table", class_=lambda x: x and "wikitable" in x)

# Print number of tables found (commented out after observing)
# print(len(wiki_tables))

# Find table with host countries and year
host_table = None
for table in wiki_tables:
    table_header = table.find_all("th") # Get all header cells
    table_header_txt = " ".join([h.text.strip().lower()
                                 for h in table_header]) # Make a list of lowercase header cells so key words can be searched for
    # Print all header strings so we can choose the most relevant table (commented out after observing)
    # print(table_header_txt)
    if "year" in table_header_txt and "host" in table_header_txt:
        host_table = table # Select the table that has "year" and "host" as headers
        break

# Print to confirm we have the right wikitable (commented out after observing)        
# print(host_table)


### Extract Hosts Data to a DataFrame and Export Raw Data to CSV

In [42]:
# Get every row from the table
rows = host_table.find_all("tr")

data = []

# Skip the first two rows (column names) and extract the rest
for row in rows[2:]:
    cols = [c.text.strip() for c 
            in row.find_all(["td", "th"])] # Extract <td> and <th> without spaces
    data.append(cols)

# Convert to DataFrame
df = pd.DataFrame(data, columns = ["Edition", "Year", "Host", "First Place", "Score", 
                                   "Runner-up", "Third Place", "Score2", "Fourth Place", "Nr. of Teams"])

# Print out to check DataFrame (commented out after observing)
# print(df)

# Save raw DataFrame as a CSV file
df.to_csv("hosts_and_years_raw.csv")


## Cleaning Hosts Data

### Check Data Type for Each Column

In [43]:
# Check data type for each column  (commented out after observing)
# df.info()

### Check for Missing Data in the Dataset

In [44]:
# Check for missing data in the dataset  (commented out after observing)
data_missing = df.isnull()
# data_missing

In [45]:
# Find missing data count per column (commented out after observing)
# df.isnull().sum()

### Drop Rows with Missing Data

##### In 1942 and 1946 no World Cup took place due to WWII
##### The 2026, 2030 and 2034 World Cups haven't taken place yet
##### There is no data for these rows so they should be dropped

#### Clean Missing Score Columns Using SQL

In [46]:
# Create a copy of the original DataFrame to prevent changing raw data
df1 = df.copy()

# Set up in-memory SQL database to allow SQL cleaning
conn = sqlite3.connect(":memory:")

# Load DataFrame into SQLite as a table
df1.to_sql("df1", conn, if_exists = "replace", index=False)

# Replace "None", "NaN" and NULL values with an empty string
df1 = conn.execute("""
    UPDATE df1
    SET Score = ""
    WHERE Score IS NULL OR Score = 'None' OR Score = 'NaN'
""")
conn.commit()

# Print the Score and Year column to check this worked
score_updated = pd.read_sql("""
    SELECT Year, Score
    FROM df1
""", conn)

# Print to check this worked (commented out after observing)
# print(score_updated.to_string(index = False))


#### Check we are Dropping the Correct Rows Using SQL

In [47]:
# Check we are dropping the correct rows
check_empty = pd.read_sql("""
    SELECT *
    FROM df1
    WHERE Score = ""
""", conn)

# Print to check (commented out after observing)
# print(check_empty.to_string(index = False))

#### Delete the Rows Using SQL

In [48]:
# Dropping the missing data
df1 = conn.execute("""
    DELETE FROM df1
    WHERE Score = ""
""")
conn.commit()

# Check this worked
rows_not_deleted = pd.read_sql("""
    SELECT *
    FROM df1
""", conn)

# Print to check (commented out after observing)
print(rows_deleted.to_string(index = False))

                  Team Appearances Record Streak Active Streak Debut Most Recent Qualification                              Best Result
                Brazil          23            23            23  1930                      2026 Champions (1958, 1962, 1970, 1994, 2002)
            Germany[a]          21            19            19  1934                      2026    Champions (1954, 1974,[b] 1990, 2014)
             Argentina          19            14            14  1930                      2026          Champions (1978,[b] 1986, 2022)
                 Italy          18            14             0  1934                      2014    Champions (1934,[b] 1938, 1982, 2006)
                Mexico          18             9             9  1930                      2026        Quarter-finals (1970,[b] 1986[b])
                 Spain          17            13            13  1934                      2026                         Champions (2010)
               England          17             8

#### Extract Relevant Columns Only Using SQL

In [10]:
# Extract "Year" and "Host" from the hosts_table
df1 = pd.read_sql("""
    SELECT Year, Host
    FROM df1
""", conn)

# Print the filtered DataFrame to check this worked (commented out after observing)
# print(df1.to_string(index=False))

## Download Elo Ratings Dataset into Pandas

##### Please refer to the README file regarding this code
##### The only complete source of Elo Ratings is a JavaScript webpage so I was not able to webscrape as I am not familiar enough with Selenium
##### Instead, this Dataset is downloaded from Kagglehub into Pandas

### Download and Locally Save the Elo Dataset

In [11]:
# Specify the file within the kagglehub dataset
target_file_path = "eloratings.csv"

# Download the specified dataset directly into a Pandas DataFrame
elo_ranking_raw = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "saifalnimri/international-football-elo-ratings",
  target_file_path,
)

# Print to check this has worked properly (commented out after observing)
# print(elo_ranking_raw)

# Save as CSV file
df.to_csv("elo_ranking_raw.csv")

/var/folders/kd/p53w9pwd7rv29w5h_tss9l580000gn/T/ipykernel_58403/3128765576.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  elo_ranking_raw = kagglehub.load_dataset(


## Cleaning Elo Data

### Check Data Type for Each Column

In [12]:
# Check data type for each column  (commented out after observing)
elo_ranking_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6678 entries, 0 to 6677
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    6678 non-null   object 
 1   team    6678 non-null   object 
 2   rating  6647 non-null   float64
 3   change  6678 non-null   int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 208.8+ KB


### Check for Missing Data in the Dataset

In [13]:
# Check for missing data in the dataset  (commented out after observing)
data_missing = elo_ranking_raw.isnull()
# data_missing

In [14]:
# Find missing data count per column (commented out after observing)
# elo_ranking_raw.isnull().sum()

### Drop rows where rating is missing using SQL

##### Please refer to the README file for the reasoning of this

#### Clean missing rating column using SQL

In [15]:
# Create a copy of the original DataFrame to prevent changing raw data
elo_ranking1 = elo_ranking_raw.copy()

# Set up in-memory SQL database to allow SQL cleaning
conn = sqlite3.connect(":memory:")

# Load DataFrame into SQLite as a table
elo_ranking1.to_sql("elo_ranking1", conn, if_exists = "replace", index=False)

# Replace "None", "NaN" and NULL values with an empty string
elo_ranking1 = conn.execute("""
    UPDATE elo_ranking1
    SET rating = ""
    WHERE rating IS NULL OR rating = 'None' OR rating = 'NaN'
""")
conn.commit()

# Print the rating, date and team columns to check this worked
rating_updated = pd.read_sql("""
    SELECT rating, date, team
    FROM elo_ranking1
""", conn)

# Print to check this worked (commented out after observing)
# print(rating_updated.to_string(index = False))


#### Check we are Dropping the Correct Rows Using SQL

In [16]:
# Check we are dropping the correct rows
check_empty = pd.read_sql("""
    SELECT *
    FROM elo_ranking1
    WHERE rating = ""
""", conn)

# Print to check (commented out after observing)
#print(check_empty.to_string(index = False))

# Note: all empty ratings are Moldova


#### Delete the Rows Using SQL

In [17]:
# Dropping the missing data
elo_ranking1 = conn.execute("""
    DELETE FROM elo_ranking1
    WHERE rating = ""
""")
conn.commit()

# Check this worked
rows_not_deleted = pd.read_sql("""
    SELECT *
    FROM elo_ranking1
""", conn)

# Print to check (commented out after observing)
# print(rows_not_deleted.to_string(index = False))

#### Check Rows were Deleted Properly

In [18]:
# Convert SQLite elo_ranking1 to pandas DataFrame
elo_ranking1 = pd.read_sql("""
    SELECT * 
    FROM elo_ranking1
""", conn)

# Return how many null in each column (commented out after observing)
# elo_ranking1.isnull().sum()

### Clean date column to be year only in elo_ranking

In [19]:
# Make copy of table to avoid changing raw data
elo_ranking2 = elo_ranking1.copy()

# Clean to year only (from mixed formatting)
elo_ranking2["date"] = pd.to_datetime(
    elo_ranking2["date"], format = "mixed"
).dt.year

# Check this worked for both date formats (commented out after observing)
print(elo_ranking2)

      date                      team  rating  change
0     1872                   England  2003.0       3
1     1872                  Scotland  1997.0      -3
2     1873                   England  2014.0      11
3     1873                  Scotland  1986.0     -11
4     1874                   England  2006.0      -8
...    ...                       ...     ...     ...
6642  2025  Northern Mariana Islands   432.0       0
6643  2025             Cocos Islands   422.0       0
6644  2025                     Palau   402.0       0
6645  2025             Eastern Samoa   389.0       0
6646  2025                   Moldova     0.0       0

[6647 rows x 4 columns]


### Check for duplicate years in each country for elo_ranking

#### Check how many duplicates are in the DataFrame

In [20]:
# Check how many duplicates are in the DataFrame
duplicates_count = elo_ranking2.duplicated(subset = ["date", "team"]).sum()
# Print to check (commented out after observing)
print(duplicates_count)

2082


### Clean to get rid of duplicates in elo_ranking

#### Create an average for the ratings column where entries have the same year and team

In [21]:
# Group together entries where date (year) and team are the same, and make the rating the average of these previous entries
elo_ranking3 = elo_ranking2.groupby(["date", "team"], as_index = False)["rating"].mean()
# Print to check (commented out after observing)
print(elo_ranking3)

      date            team       rating
0     1872         England  2003.000000
1     1872        Scotland  1997.000000
2     1873         England  2014.000000
3     1873        Scotland  1986.000000
4     1874         England  2006.000000
...    ...             ...          ...
4560  2025  Western Sahara   996.000000
4561  2025           Yemen  1135.666667
4562  2025          Zambia  1364.000000
4563  2025        Zanzibar  1319.000000
4564  2025        Zimbabwe  1366.250000

[4565 rows x 3 columns]


#### Check this worked by re-checking for duplicates

In [22]:
# Check how many duplicates are in the DataFrame
duplicates_count = elo_ranking3.duplicated(subset = ["date", "team"]).sum()
# Print to check (commented out after observing)
print(duplicates_count)

0


## Webscrape the Names of All Participant Countries

### Fetch and Locate Hosts Data

In [23]:
# Fetch HTML from Wikipedia page
participants_url = "https://en.wikipedia.org/wiki/National_team_appearances_in_the_FIFA_World_Cup"

# Create request with custom User Agent to avoid having the request blocked
req = urllib.request.Request(participants_url, headers={'User-Agent' : "Magic Browser"}) 

# Retrieve page content
con = urllib.request.urlopen(req)

# Parse the HTML using BeautifulSoup
soup = BeautifulSoup(con.read(), "html.parser")

# Find all tables with the class "wikitable"
wiki_table_participants = soup.find_all("table", class_=lambda x: x and "wikitable" in x)

# Print number of tables found (commented out after observing)
#print(len(wiki_table_participants))

# Find table with host countries and year
participants_table = None
for table in wiki_table_participants:
    table_header = table.find_all("th") # Get all header cells
    table_header_txt = " ".join([h.text.strip().lower()
                                 for h in table_header]) # Make a list of lowercase header cells so key words can be searched for
    # Print all header strings so we can choose the most relevant table (commented out after observing)
    # print(table_header_txt)
    if "team" in table_header_txt and "appearances" in table_header_txt:
        participants_table = table # Select the table that has "team" and "appearances" as headers
        break

# Print to confirm we have the right wikitable (commented out after observing)        
# print(participants_table)

### Extract Hosts Data to a DataFrame and Export Raw Data to CSV

In [24]:
# Get every row from the table
rows = participants_table.find_all('tr')

data = []

# Skip the first row (column names) and extract the rest
for row in rows[1:]:
    cols = [c.text.strip() for c 
            in row.find_all(['td', 'th'])] # Extract <td> and <th> without spaces
    data.append(cols)

# Convert to DataFrame
participants = pd.DataFrame(data, columns = ["Team", "Appearances", "Record Streak", "Active Streak", "Debut", 
                                   "Most Recent Qualification", "Best Result"])

# Print out to check DataFrame (commented out after observing)
# print(participants)

# Save raw DataFrame as a CSV file
participants.to_csv("participants_raw.csv")


## Cleaning Participant Country DataFrame

### Delete the countries that have not yet debuted (debut in 2026)

In [25]:
# Create a copy of the original DataFrame to prevent changing raw data
participants1 = participants.copy()

# Set up in-memory SQL database to allow SQL cleaning
conn = sqlite3.connect(":memory:")

# Load DataFrame into SQLite as a table
participants1.to_sql("participants1", conn, if_exists="replace", index=False)

# Dropping the missing data
participants1 = conn.execute("""
    DELETE FROM participants1
    WHERE Debut = 2026
""")
conn.commit()

# Check this worked
rows_deleted = pd.read_sql("""
    SELECT *
    FROM participants1
""", conn)

# Print to check (commented out after observing)
# print(rows_deleted.to_string(index=False))


## Webscrape Stats Per Country Per Year

In [35]:
# Extract Worlc Cup  years from Hosts DataFrame
years = df1["Year"].tolist()

# Check this worked (commented out after observing)
print(years)

years.remove("1978")
years.remove("1990")
years.remove("2002")

print(years)


all_rows = []


hosts_url = "https://en.wikipedia.org/wiki/FIFA_World_Cup"
req = urllib.request.Request(hosts_url, headers={'User-Agent' : "Magic Browser"}) 
con = urllib.request.urlopen(req)

soup = BeautifulSoup(con.read(), "html.parser")

# Find all tables
wiki_tables = soup.find_all("table", class_=lambda x: x and "wikitable" in x)
#print(len(wiki_tables))
# Find table with host countrie(s) and year
host_table = None
for table in wiki_tables:
    table_header = table.find_all("th") # Get all header cells
    table_header_txt = " ".join([h.text.strip().lower()
                                 for h in table_header]) # Make a list of header cells so key words can be searched for
    print(table_header_txt)
    if "year" in table_header_txt and "host" in table_header_txt:
        host_table = table
        break


for year in years:
    points_url = f"https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup"
    print("URL:",points_url)
    req = urllib.request.Request(points_url, headers={'User-Agent' : "Magic Browser"}) 
    con = urllib.request.urlopen(req)

    soup = BeautifulSoup(con.read(), "html.parser")

    wiki_table_points = soup.find_all("table", class_="wikitable")

    points_table = None
    for table in wiki_table_points:

        if points_table is not None:
            break
        header = table.find("tr")
        if header:
            first_th = header.find("th")
            if first_th:
                abbr = first_th.find("abbr")
                if abbr and "final ranking" == abbr.get("title", "").lower():
                    points_table = table
            # if points_table is None:
            #     print ("LENGHT:", len(table.find_all("tr")))
            if points_table is None and len(table.find_all("tr")) > 8:
                last_th = header.find_all("th")[-1]
                if last_th:
                    text = last_th.text.strip()

                    print ("TEXT:", text)
                    if text and "Result" == text:
                        points_table = table

    if points_table is not None:
        # Extract rows
        rows = points_table.find_all("tr")
    
        for row in rows[1:]:  # skip header row
            cols = row.find_all(["td", "th"]) # Some years have 13 columns - drop column 13
            print("COLLEN: ", len(cols))
            cols = [col.get_text(strip=True) for col in cols]
    
            if cols:
                all_rows.append([year] + cols[:11])
    
        # create dataframe
        total_points_table_raw = pd.DataFrame(all_rows, columns = ["Year", "Ranking", "Team", "Group", "Matches Played", "Matches Won", 
                                           "Matches Drawn", "Matches Lost", "Goals For", "Goals Against",
                                           "Goal Difference", "Points"])
    
        #print(total_points_table)

    else:
        print(f"TABLE NOT FOUND{year}")                        

print(total_points_table_raw.to_string(index=False))

# Save raw DataFrame as a CSV file
total_points_table_raw.to_csv("total_points_table_raw.csv")


['1930', '1934', '1938', '1950', '1954', '1958', '1962', '1966', '1970', '1974', '1978', '1982', '1986', '1990', '1994', '1998', '2002', '2006', '2010', '2014', '2018', '2022']
['1930', '1934', '1938', '1950', '1954', '1958', '1962', '1966', '1970', '1974', '1982', '1986', '1994', '1998', '2006', '2010', '2014', '2018', '2022']
confederation times hosted hosts upcoming hosts
ed. year host first place game third place game teams champion score runner-up third place score fourth place 1 2 3 – 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25
URL: https://en.wikipedia.org/wiki/1930_FIFA_World_Cup
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  1
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
URL: https://en.wikipedia.org/wiki/1934_FIFA_World_Cup
TEXT: Bologna
COLLEN:  10
COLLEN:  10
COLLEN:  10
COLLEN:  10
COLLEN:  1
COLLEN:  10
COLLEN:  10
COLLEN:  10
COLLEN:  10
COLLEN:  1
COLLEN:  10
COLLEN:  9
COLLEN:  9
C

In [38]:
# Extract Worlc Cup  years from Hosts DataFrame
years = ["1978", "1990", "2002"]

# Check this worked (commented out after observing)
print(years)


all_rows = []


hosts_url = "https://en.wikipedia.org/wiki/FIFA_World_Cup"
req = urllib.request.Request(hosts_url, headers={'User-Agent' : "Magic Browser"}) 
con = urllib.request.urlopen(req)

soup = BeautifulSoup(con.read(), "html.parser")

# Find all tables
wiki_tables = soup.find_all("table", class_=lambda x: x and "wikitable" in x)
#print(len(wiki_tables))
# Find table with host countrie(s) and year
host_table = None
for table in wiki_tables:
    table_header = table.find_all("th") # Get all header cells
    table_header_txt = " ".join([h.text.strip().lower()
                                 for h in table_header]) # Make a list of header cells so key words can be searched for
    print(table_header_txt)
    if "year" in table_header_txt and "host" in table_header_txt:
        host_table = table
        break


for year in years:
    points_url = f"https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup"
    print("URL:",points_url)
    req = urllib.request.Request(points_url, headers={'User-Agent' : "Magic Browser"}) 
    con = urllib.request.urlopen(req)

    soup = BeautifulSoup(con.read(), "html.parser")

    wiki_table_points = soup.find_all("table", class_="wikitable")

    points_table = None
    for table in wiki_table_points:

        if points_table is not None:
            break
        header = table.find("tr")
        if header:
            first_th = header.find("th")
            if first_th:
                abbr = first_th.find("abbr")
                if abbr and "final ranking" == abbr.get("title", "").lower():
                    points_table = table
            # if points_table is None:
            #     print ("LENGHT:", len(table.find_all("tr")))
            if points_table is None and len(table.find_all("tr")) > 8:
                last_th = header.find_all("th")[-1]
                if last_th:
                    text = last_th.text.strip()

                    print ("TEXT:", text)
                    if text and "Result" == text:
                        points_table = table

    if points_table is not None:
        # Extract rows
        rows = points_table.find_all("tr")
    
        for row in rows[1:]:  # skip header row
            cols = row.find_all(["td", "th"]) # Some years have 13 columns - drop column 13
            print("COLLEN: ", len(cols))
            cols = [col.get_text(strip=True) for col in cols]
    
            if cols:
                all_rows.append([year] + cols[:12])
    
        # create dataframe
        total_points_table_raw = pd.DataFrame(all_rows, columns = ["Year", "Ranking", "Group", "Team", "Matches Played", "Matches Won", 
                                           "Matches Drawn", "Matches Lost", "Goals For", "Goals Against",
                                           "Goal Difference", "Points", "Result"])
    
        #print(total_points_table)

    else:
        print(f"TABLE NOT FOUND{year}")                        

print(total_points_table_raw.to_string(index=False))

# Save raw DataFrame as a CSV file
total_points_table_raw.to_csv("total_points_table_raw.csv")


['1978', '1990', '2002']
confederation times hosted hosts upcoming hosts
ed. year host first place game third place game teams champion score runner-up third place score fourth place 1 2 3 – 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25
URL: https://en.wikipedia.org/wiki/1978_FIFA_World_Cup
TEXT: Córdoba City, Córdoba
TEXT: Result
COLLEN:  12
COLLEN:  12
COLLEN:  12
COLLEN:  12
COLLEN:  12
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  12
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
URL: https://en.wikipedia.org/wiki/1990_FIFA_World_Cup
TEXT: Naples
TEXT: Result
COLLEN:  12
COLLEN:  12
COLLEN:  12
COLLEN:  12
COLLEN:  12
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  12
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  12
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
COLLEN:  11
URL: https://en.wikipedia.org/wiki/2002_FIFA_World_Cup
TEXT: South Korea
TEXT: Result
COLLEN